In [ ]:
import re
import wave
from pathlib import Path

import pandas as pd
import tgt


def load_table(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    raise ValueError(f"Unsupported metadata format: {suffix}")


def get_wav_duration(file_path):
    try:
        with wave.open(str(file_path), "rb") as f:
            return f.getnframes() / float(f.getframerate())
    except Exception:
        return 0.0


def get_valid_wavs_with_textgrids(folder):
    """
    Return only .wav files that have a matching .TextGrid in the same folder.
    Match is by stem, e.g.:
      abc.wav <-> abc.TextGrid
    """
    valid_wavs = []
    for wav_path in folder.rglob("*.wav"):
        textgrid_path = wav_path.with_suffix(".TextGrid")
        if textgrid_path.exists():
            valid_wavs.append(wav_path)
    return valid_wavs


def count_words_in_textgrid(textgrid_path):
    """
    Count non-empty word intervals from the MFA 'words' tier.
    """
    try:
        tg = tgt.read_textgrid(str(textgrid_path))
        tier = tg.get_tier_by_name("words")
    except Exception:
        return 0

    count = 0
    for interval in tier.intervals:
        text = str(interval.text).strip()
        if text and text.lower() not in {"sil", "sp", "spn", "<sil>", "silence"}:
            count += 1
    return count


def estimate_cm_windows(num_words, max_context_words=10):
    """
    Estimate the number of CM windows produced by the dataset expansion
    for a single utterance with num_words labeled words.
    """
    n = int(num_words) if pd.notna(num_words) else 0
    k = int(max_context_words)
    if n <= 0:
        return 0
    return sum(max(0, n - c + 1) for c in range(1, k + 1))


def get_folder_stats(folder):
    """
    Sum duration, word count, and utterance count across all valid wav/TextGrid pairs.
    Each valid wav/TextGrid pair with at least one counted word is treated as one utterance.
    """
    valid_wavs = get_valid_wavs_with_textgrids(folder)

    total_duration = 0.0
    total_words = 0
    total_utterances = 0

    for wav_path in valid_wavs:
        tg_path = wav_path.with_suffix(".TextGrid")

        dur = get_wav_duration(wav_path)
        n_words = count_words_in_textgrid(tg_path)

        if dur > 0 and n_words > 0:
            total_duration += dur
            total_words += n_words
            total_utterances += 1

    return total_duration, total_words, total_utterances


def normalize_pid(value):
    s = str(value).strip()
    m = re.search(r"(\d{3})$", s)
    if m:
        return m.group(1)
    return s[-3:]


def extract_base_pid_from_text(text):
    m = re.search(r"(\d{3})", str(text))
    return m.group(1) if m else None


def extract_session_number(value):
    s = str(value).strip()

    patterns = [
        r"ses[-_]?(\d+)",
        r"session[-_]?(\d+)",
        r"(\d+)$",
    ]

    for pattern in patterns:
        m = re.search(pattern, s, flags=re.IGNORECASE)
        if m:
            return int(m.group(1))

    return None


def normalize_gender(value):
    if pd.isna(value):
        return pd.NA

    s = str(value).strip().upper()
    return s if s else pd.NA


def format_score(score_value, scale_name="CDS"):
    if pd.isna(score_value):
        return f"{scale_name} NA"

    score_value = float(score_value)
    if score_value.is_integer():
        return f"{scale_name} {int(score_value)}"
    return f"{scale_name} {score_value:.2f}".rstrip("0").rstrip(".")


def format_panss_score(score_value):
    if pd.isna(score_value):
        return "PANSS8 NA"

    score_value = float(score_value)
    if score_value.is_integer():
        return f"PANSS8 {int(score_value)}"
    return f"PANSS8 {score_value:.2f}".rstrip("0").rstrip(".")


def score_to_label_depression(score, threshold=6):
    return "D" if score >= threshold else "H"


def cds_to_depression_severity(score):
    if pd.isna(score):
        return pd.NA

    score = float(score)
    if score < 6:
        return 0
    elif score <= 9:
        return 1
    else:
        return 2


def compute_cds_row_score(meta):
    if "CDS_Total" not in meta.columns:
        raise ValueError("This script requires a 'CDS_Total' column.")

    meta = meta.copy()

    cds_total = pd.to_numeric(meta["CDS_Total"], errors="coerce")
    cds_total_idx = meta.columns.get_loc("CDS_Total")

    prior_cols = list(meta.columns[max(0, cds_total_idx - 9):cds_total_idx])
    fallback_cols = [c for c in prior_cols if str(c).startswith("CDS")]

    if cds_total.isna().any() and not fallback_cols:
        raise ValueError(
            "CDS_Total has NaNs, but no fallback CDS columns were found among the prior 9 columns."
        )

    if fallback_cols:
        fallback_sum = (
            meta[fallback_cols]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0)
            .sum(axis=1)
        )
    else:
        fallback_sum = pd.Series(0, index=meta.index, dtype=float)

    meta["cds_score"] = cds_total.where(~cds_total.isna(), fallback_sum)
    meta["depression_severity"] = meta["cds_score"].apply(cds_to_depression_severity)
    return meta


def compute_psychosis_columns(meta):
    meta = meta.copy()

    if "phenotype" not in meta.columns:
        raise ValueError("This script requires a 'phenotype' column.")

    phenotype = meta["phenotype"].astype(str).str.strip().str.upper()
    meta["label_psychosis"] = phenotype.eq("HC").map({True: "N", False: "Y"})

    panss8_cols = [c for c in meta.columns if str(c).startswith("PANSS8")]
    if len(panss8_cols) != 8:
        raise ValueError(
            f"Expected exactly 8 PANSS8* columns, found {len(panss8_cols)}: {panss8_cols}"
        )

    panss8 = meta[panss8_cols].apply(pd.to_numeric, errors="coerce")
    remission_from_panss = panss8.le(3).all(axis=1).map({True: "Y", False: "N"})

    meta["psychosis_remission"] = remission_from_panss
    meta.loc[meta["label_psychosis"] == "N", "psychosis_remission"] = "NA"

    if "panss-total" not in meta.columns:
        raise ValueError("This script requires a 'panss-total' column.")

    meta["panss_total_numeric"] = pd.to_numeric(meta["panss-total"], errors="coerce")
    meta["score_psychosis"] = meta["panss_total_numeric"].apply(format_panss_score)

    return meta


def build_lookup_tables(meta, participant_col="participant_id", session_col="session_id"):
    meta = meta.copy()

    if "sex" not in meta.columns:
        raise ValueError("This script requires a 'sex' column.")
    if "age" not in meta.columns:
        raise ValueError("This script requires an 'age' column.")

    meta["gender"] = meta["sex"].apply(normalize_gender)
    meta["age_numeric"] = pd.to_numeric(meta["age"], errors="coerce")
    meta["raw_base_pid"] = meta[participant_col].apply(normalize_pid)
    meta["session_num"] = meta[session_col].apply(extract_session_number)

    if meta["session_num"].isna().any():
        bad_rows = meta.loc[
            meta["session_num"].isna(),
            [participant_col, session_col]
        ].head(10)
        raise ValueError(
            "Could not parse session number for some rows. Example rows:\n"
            f"{bad_rows}"
        )

    meta["session_num"] = meta["session_num"].astype(int)

    meta["psychosis_numeric"] = meta["label_psychosis"].map({"Y": 1, "N": 0})
    meta["remission_numeric"] = meta["psychosis_remission"].map({"Y": 1, "N": 0})
    meta["panss_total_numeric"] = pd.to_numeric(meta["panss_total_numeric"], errors="coerce")

    per_session = (
        meta.groupby(["raw_base_pid", "session_num"], as_index=False)
        .agg(
            cds_score=("cds_score", "mean"),
            depression_severity=("depression_severity", "mean"),
            psychosis_numeric=("psychosis_numeric", "max"),
            remission_numeric=("remission_numeric", "min"),
            panss_total_numeric=("panss_total_numeric", "mean"),
            gender=("gender", "first"),
            age_numeric=("age_numeric", "mean"),
        )
        .sort_values(["raw_base_pid", "session_num"])
    )

    baseline_map = {}
    followup_map = {}

    for raw_pid, subdf in per_session.groupby("raw_base_pid"):
        baseline_rows = subdf[subdf["session_num"] == 1]
        followup_rows = subdf[subdf["session_num"] > 1]

        if not baseline_rows.empty:
            row = baseline_rows.iloc[0]
            baseline_cds = float(row["cds_score"])

            baseline_map[raw_pid] = {
                "score_depression_value": baseline_cds,
                "label_depression": score_to_label_depression(baseline_cds),
                "score_depression": format_score(baseline_cds, "CDS"),
                "depression_severity": int(row["depression_severity"]) if pd.notna(row["depression_severity"]) else pd.NA,
                "label_psychosis": "Y" if row["psychosis_numeric"] >= 1 else "N",
                "psychosis_remission": (
                    "NA" if row["psychosis_numeric"] < 1
                    else ("Y" if row["remission_numeric"] >= 1 else "N")
                ),
                "score_psychosis": format_panss_score(row["panss_total_numeric"]),
                "gender": row["gender"],
                "age": row["age_numeric"],
            }

        if not followup_rows.empty:
            mean_cds = float(followup_rows["cds_score"].mean())
            mean_panss = followup_rows["panss_total_numeric"].mean()

            followup_gender = (
                followup_rows["gender"].dropna().iloc[0]
                if not followup_rows["gender"].dropna().empty
                else pd.NA
            )
            followup_age = followup_rows["age_numeric"].mean()

            followup_map[raw_pid] = {
                "score_depression_value": mean_cds,
                "label_depression": score_to_label_depression(mean_cds),
                "score_depression": format_score(mean_cds, "CDS"),
                "depression_severity": cds_to_depression_severity(mean_cds),
                "label_psychosis": "Y" if followup_rows["psychosis_numeric"].max() >= 1 else "N",
                "psychosis_remission": (
                    "NA"
                    if followup_rows["psychosis_numeric"].max() < 1
                    else ("Y" if followup_rows["remission_numeric"].min() >= 1 else "N")
                ),
                "score_psychosis": format_panss_score(mean_panss),
                "gender": followup_gender,
                "age": followup_age,
            }

    return baseline_map, followup_map


def collect_zero_folder_speakers(zero_root, baseline_map, dataset_name):
    rows = []

    if not zero_root.exists():
        print(f"Warning: {zero_root} does not exist. Skipping 0-folder scan.")
        return rows

    for speaker_folder in sorted(zero_root.iterdir()):
        if not speaker_folder.is_dir():
            continue

        participant_id = speaker_folder.name
        raw_base_pid = extract_base_pid_from_text(participant_id)

        if raw_base_pid is None:
            print(f"Skipping {participant_id}: could not extract 3-digit participant id.")
            continue

        if raw_base_pid not in baseline_map:
            print(f"Skipping {participant_id}: no ses-1 metadata found for {raw_base_pid}.")
            continue

        duration, num_words, num_utterances = get_folder_stats(speaker_folder)

        if duration <= 0 or num_words <= 0 or num_utterances <= 0:
            print(f"Skipping {participant_id}: no usable .wav/.TextGrid pairs with words found.")
            continue

        info = baseline_map[raw_base_pid]

        if pd.isna(info["gender"]):
            print(f"Skipping {participant_id}: missing gender for {raw_base_pid}.")
            continue

        rows.append(
            {
                "participant_id": participant_id,
                "dataset": dataset_name,
                "base_pid": raw_base_pid,
                "duration": duration,
                "num_words": int(num_words),
                "num_utterances": int(num_utterances),
                "gender": info["gender"],
                "age": info["age"],
                "label_depression": info["label_depression"],
                "score_depression": info["score_depression"],
                "depression_severity": info["depression_severity"],
                "label_psychosis": info["label_psychosis"],
                "psychosis_remission": info["psychosis_remission"],
                "score_psychosis": info["score_psychosis"],
            }
        )

    return rows


def collect_one_folder_speakers(one_root, followup_map, dataset_name):
    rows = []

    if not one_root.exists():
        print(f"Warning: {one_root} does not exist. Skipping 1-folder scan.")
        return rows

    seen = set()

    for speaker_folder in sorted(one_root.glob("*/*")):
        if not speaker_folder.is_dir():
            continue

        participant_id = speaker_folder.name
        if participant_id in seen:
            continue
        seen.add(participant_id)

        raw_base_pid = extract_base_pid_from_text(participant_id)
        if raw_base_pid is None:
            print(f"Skipping {participant_id}: could not extract 3-digit participant id.")
            continue

        if raw_base_pid not in followup_map:
            print(f"Skipping {participant_id}: no follow-up metadata found for {raw_base_pid}.")
            continue

        duration, num_words, num_utterances = get_folder_stats(speaker_folder)

        if duration <= 0 or num_words <= 0 or num_utterances <= 0:
            print(f"Skipping {participant_id}: no usable .wav/.TextGrid pairs with words found.")
            continue

        info = followup_map[raw_base_pid]

        if pd.isna(info["gender"]):
            print(f"Skipping {participant_id}: missing gender for {raw_base_pid}.")
            continue

        rows.append(
            {
                "participant_id": participant_id,
                "dataset": dataset_name,
                "base_pid": raw_base_pid,
                "duration": duration,
                "num_words": int(num_words),
                "num_utterances": int(num_utterances),
                "gender": info["gender"],
                "age": info["age"],
                "label_depression": info["label_depression"],
                "score_depression": info["score_depression"],
                "depression_severity": info["depression_severity"],
                "label_psychosis": info["label_psychosis"],
                "psychosis_remission": info["psychosis_remission"],
                "score_psychosis": info["score_psychosis"],
            }
        )

    return rows


def assign_grouped_balanced_splits(
    df,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    word_weight=15.0,
    utterance_weight=12.0,
    gender_weight=8.0,
    depression_weight=6.0,
    psychosis_weight=4.0,
    depression_severity_weight=2.0,
    psychosis_remission_weight=1.0,
    min_dep_groups_val=4,
    min_dep_groups_test=4,
    min_psy_groups_val=8,
    min_psy_groups_test=8,
    soft_cap_factor=1.10,
    overflow_penalty=1e6,
):
    """
    Assign splits by (dataset, base_pid) so the same underlying speaker stays in
    the same split across 0 and 1 branches.

    Split balance uses utterance counts, word counts, and the existing clinical
    label logic.
    """
    splits = ["train", "val", "test"]
    split_ratios = {"train": train_ratio, "val": val_ratio, "test": test_ratio}

    gender_labels = sorted(df["gender"].dropna().unique())
    depression_labels = sorted(df["label_depression"].dropna().unique())
    psychosis_labels = sorted(df["label_psychosis"].dropna().unique())
    depression_severity_labels = sorted(df["depression_severity"].dropna().unique())
    psychosis_remission_labels = sorted(
        [x for x in df["psychosis_remission"].dropna().unique() if x in {"Y", "N"}]
    )

    speaker_groups = []
    for (dataset, base_pid), subdf in df.groupby(["dataset", "base_pid"]):
        gender_word = {label: 0.0 for label in gender_labels}
        dep_word = {label: 0.0 for label in depression_labels}
        psy_word = {label: 0.0 for label in psychosis_labels}
        depsev_word = {label: 0.0 for label in depression_severity_labels}
        rem_word = {label: 0.0 for label in psychosis_remission_labels}

        for label, dsub in subdf.groupby("gender"):
            gender_word[label] = float(dsub["num_words"].sum())

        for label, dsub in subdf.groupby("label_depression"):
            dep_word[label] = float(dsub["num_words"].sum())

        for label, dsub in subdf.groupby("label_psychosis"):
            psy_word[label] = float(dsub["num_words"].sum())

        for label, dsub in subdf.groupby("depression_severity"):
            depsev_word[label] = float(dsub["num_words"].sum())

        for label, dsub in subdf.groupby("psychosis_remission"):
            if label in {"Y", "N"}:
                rem_word[label] = float(dsub["num_words"].sum())

        total_words = float(subdf["num_words"].sum())
        total_utterances = float(subdf["num_utterances"].sum())

        speaker_groups.append(
            {
                "dataset": dataset,
                "base_pid": base_pid,
                "total_words": total_words,
                "total_utterances": total_utterances,
                "gender_word": gender_word,
                "dep_word": dep_word,
                "psy_word": psy_word,
                "depsev_word": depsev_word,
                "rem_word": rem_word,
                "has_dep": bool((subdf["label_depression"] == "D").any()),
                "has_psy": bool((subdf["label_psychosis"] == "Y").any()),
            }
        )

    speaker_groups.sort(
        key=lambda x: (x["total_words"], x["total_utterances"]),
        reverse=True,
    )

    total_words = float(df["num_words"].sum())
    total_utterances = float(df["num_utterances"].sum())

    total_gender = {
        label: float(df.loc[df["gender"] == label, "num_words"].sum())
        for label in gender_labels
    }
    total_dep = {
        label: float(df.loc[df["label_depression"] == label, "num_words"].sum())
        for label in depression_labels
    }
    total_psy = {
        label: float(df.loc[df["label_psychosis"] == label, "num_words"].sum())
        for label in psychosis_labels
    }
    total_depsev = {
        label: float(df.loc[df["depression_severity"] == label, "num_words"].sum())
        for label in depression_severity_labels
    }
    total_rem = {
        label: float(df.loc[df["psychosis_remission"] == label, "num_words"].sum())
        for label in psychosis_remission_labels
    }

    target_total_words = {
        split: split_ratios[split] * total_words
        for split in splits
    }
    target_total_utterances = {
        split: split_ratios[split] * total_utterances
        for split in splits
    }

    target_gender = {
        split: {label: split_ratios[split] * total_gender[label] for label in gender_labels}
        for split in splits
    }
    target_dep = {
        split: {label: split_ratios[split] * total_dep[label] for label in depression_labels}
        for split in splits
    }
    target_psy = {
        split: {label: split_ratios[split] * total_psy[label] for label in psychosis_labels}
        for split in splits
    }
    target_depsev = {
        split: {label: split_ratios[split] * total_depsev[label] for label in depression_severity_labels}
        for split in splits
    }
    target_rem = {
        split: {label: split_ratios[split] * total_rem[label] for label in psychosis_remission_labels}
        for split in splits
    }

    current_total_words = {split: 0.0 for split in splits}
    current_total_utterances = {split: 0.0 for split in splits}

    current_gender = {split: {label: 0.0 for label in gender_labels} for split in splits}
    current_dep = {split: {label: 0.0 for label in depression_labels} for split in splits}
    current_psy = {split: {label: 0.0 for label in psychosis_labels} for split in splits}
    current_depsev = {split: {label: 0.0 for label in depression_severity_labels} for split in splits}
    current_rem = {split: {label: 0.0 for label in psychosis_remission_labels} for split in splits}

    dep_group_counts = {"val": 0, "test": 0}
    psy_group_counts = {"val": 0, "test": 0}

    assignment = {}

    def add_to_split(split, speaker):
        assignment[(speaker["dataset"], speaker["base_pid"])] = split

        current_total_words[split] += speaker["total_words"]
        current_total_utterances[split] += speaker["total_utterances"]

        for label in gender_labels:
            current_gender[split][label] += speaker["gender_word"].get(label, 0.0)

        for label in depression_labels:
            current_dep[split][label] += speaker["dep_word"].get(label, 0.0)

        for label in psychosis_labels:
            current_psy[split][label] += speaker["psy_word"].get(label, 0.0)

        for label in depression_severity_labels:
            current_depsev[split][label] += speaker["depsev_word"].get(label, 0.0)

        for label in psychosis_remission_labels:
            current_rem[split][label] += speaker["rem_word"].get(label, 0.0)

        if split in {"val", "test"}:
            if speaker["has_dep"]:
                dep_group_counts[split] += 1
            if speaker["has_psy"]:
                psy_group_counts[split] += 1

    def over_soft_cap(split, speaker):
        if split == "train":
            return False

        after_words = current_total_words[split] + speaker["total_words"]
        after_utterances = current_total_utterances[split] + speaker["total_utterances"]

        return (
            after_words > soft_cap_factor * target_total_words[split]
            or after_utterances > soft_cap_factor * target_total_utterances[split]
        )

    def candidate_cost(split, speaker):
        eps = 1e-8
        cost = 0.0

        after_words = current_total_words[split] + speaker["total_words"]
        after_utterances = current_total_utterances[split] + speaker["total_utterances"]

        target_words = target_total_words[split] + eps
        target_utterances = target_total_utterances[split] + eps

        if split != "train":
            if after_words > soft_cap_factor * target_words:
                cost += overflow_penalty * ((after_words / target_words) - soft_cap_factor)

            if after_utterances > soft_cap_factor * target_utterances:
                cost += overflow_penalty * ((after_utterances / target_utterances) - soft_cap_factor)

        cost += word_weight * (((after_words - target_words) / target_words) ** 2)
        cost += utterance_weight * (((after_utterances - target_utterances) / target_utterances) ** 2)

        for label in gender_labels:
            target = target_gender[split][label] + eps
            after = current_gender[split][label] + speaker["gender_word"].get(label, 0.0)
            cost += gender_weight * (((after - target) / target) ** 2)

        for label in depression_labels:
            target = target_dep[split][label] + eps
            after = current_dep[split][label] + speaker["dep_word"].get(label, 0.0)
            cost += depression_weight * (((after - target) / target) ** 2)

        for label in psychosis_labels:
            target = target_psy[split][label] + eps
            after = current_psy[split][label] + speaker["psy_word"].get(label, 0.0)
            cost += psychosis_weight * (((after - target) / target) ** 2)

        for label in depression_severity_labels:
            target = target_depsev[split][label] + eps
            after = current_depsev[split][label] + speaker["depsev_word"].get(label, 0.0)
            cost += depression_severity_weight * (((after - target) / target) ** 2)

        for label in psychosis_remission_labels:
            target = target_rem[split][label] + eps
            after = current_rem[split][label] + speaker["rem_word"].get(label, 0.0)
            cost += psychosis_remission_weight * (((after - target) / target) ** 2)

        return cost

    unassigned = []
    for speaker in speaker_groups:
        placed = False

        if speaker["has_dep"]:
            candidates = []
            if dep_group_counts["val"] < min_dep_groups_val and not over_soft_cap("val", speaker):
                candidates.append("val")
            if dep_group_counts["test"] < min_dep_groups_test and not over_soft_cap("test", speaker):
                candidates.append("test")

            if candidates:
                best_split = min(candidates, key=lambda s: candidate_cost(s, speaker))
                add_to_split(best_split, speaker)
                placed = True

        if not placed:
            unassigned.append(speaker)

    still_unassigned = []
    for speaker in unassigned:
        placed = False

        if speaker["has_psy"]:
            candidates = []
            if psy_group_counts["val"] < min_psy_groups_val and not over_soft_cap("val", speaker):
                candidates.append("val")
            if psy_group_counts["test"] < min_psy_groups_test and not over_soft_cap("test", speaker):
                candidates.append("test")

            if candidates:
                best_split = min(candidates, key=lambda s: candidate_cost(s, speaker))
                add_to_split(best_split, speaker)
                placed = True

        if not placed:
            still_unassigned.append(speaker)

    for speaker in still_unassigned:
        candidates = [
            s for s in splits
            if not over_soft_cap(s, speaker) or s == "train"
        ]
        if not candidates:
            candidates = ["train"]

        best_split = min(candidates, key=lambda s: candidate_cost(s, speaker))
        add_to_split(best_split, speaker)

    out = df.copy()
    out["split"] = out.apply(
        lambda row: assignment[(row["dataset"], row["base_pid"])],
        axis=1,
    )
    return out

def create_balanced_splits_cds_specific(
    mfa_root,
    metadata_path,
    output_csv,
    dataset_name,
    participant_col="participant_id",
    session_col="session_id",
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
):
    meta = load_table(metadata_path)
    meta = compute_cds_row_score(meta)
    meta = compute_psychosis_columns(meta)

    baseline_map, followup_map = build_lookup_tables(
        meta,
        participant_col=participant_col,
        session_col=session_col,
    )

    root = Path(mfa_root)

    print("Scanning 0-folder speakers...")
    zero_rows = collect_zero_folder_speakers(
        root / "0",
        baseline_map,
        dataset_name,
    )

    print("Scanning 1-folder speakers...")
    one_rows = collect_one_folder_speakers(
        root / "1",
        followup_map,
        dataset_name,
    )

    df = pd.DataFrame(zero_rows + one_rows)

    if df.empty:
        raise ValueError("No usable speaker folders were found.")

    out_df = assign_grouped_balanced_splits(
        df,
        train_ratio=train_ratio,
        val_ratio=val_ratio,
        test_ratio=test_ratio,
        word_weight=12.0,
        utterance_weight=12.0,
        gender_weight=8.0,
        depression_weight=4.0,
        psychosis_weight=3.0,
        depression_severity_weight=2.0,
        psychosis_remission_weight=1.0,
    )

    out_df = out_df[
        [
            "participant_id",
            "base_pid",
            "dataset",
            "split",
            "gender",
            "age",
            "label_depression",
            "score_depression",
            "depression_severity",
            "label_psychosis",
            "psychosis_remission",
            "score_psychosis",
            "duration",
            "num_utterances",
            "num_words",
        ]
    ].sort_values(["split", "dataset", "participant_id"])

    out_df.to_csv(output_csv, index=False)

    print("Done.")

    print("\nCounts by split:")
    print(out_df.groupby("split")["participant_id"].count())

    print("\nCounts by split and gender:")
    print(out_df.groupby(["split", "gender"])["participant_id"].count())

    print("\nCounts by split and depression label:")
    print(out_df.groupby(["split", "label_depression"])["participant_id"].count())

    print("\nCounts by split and psychosis label:")
    print(out_df.groupby(["split", "label_psychosis"])["participant_id"].count())

    print("\nCounts by split and psychosis remission:")
    print(out_df.groupby(["split", "psychosis_remission"])["participant_id"].count())

    print("\nCounts by split and depression severity:")
    print(out_df.groupby(["split", "depression_severity"])["participant_id"].count())

    print("\nCounts by split, depression label, and gender:")
    print(out_df.groupby(["split", "label_depression", "gender"])["participant_id"].count())

    print("\nCounts by split, psychosis label, and gender:")
    print(out_df.groupby(["split", "label_psychosis", "gender"])["participant_id"].count())

    print("\nDuration by split:")
    print(out_df.groupby("split")["duration"].sum())

    print("\nUtterance counts by split:")
    print(out_df.groupby("split")["num_utterances"].sum())

    print("\nWord counts by split:")
    print(out_df.groupby("split")["num_words"].sum())

    print("\nDuration by split and depression label:")
    print(out_df.groupby(["split", "label_depression"])["duration"].sum())

    print("\nDuration by split and psychosis label:")
    print(out_df.groupby(["split", "label_psychosis"])["duration"].sum())

    print("\nWord counts by split and depression label:")
    print(out_df.groupby(["split", "label_depression"])["num_words"].sum())

    print("\nWord counts by split and psychosis label:")
    print(out_df.groupby(["split", "label_psychosis"])["num_words"].sum())

    print("\nUtterance counts by split and depression label:")
    print(out_df.groupby(["split", "label_depression"])["num_utterances"].sum())

    print("\nUtterance counts by split and psychosis label:")
    print(out_df.groupby(["split", "label_psychosis"])["num_utterances"].sum())

    print("\nAge summary by split:")
    age_summary = out_df.groupby("split")["age"].agg(["min", "max", "mean", "median"])
    print(age_summary)

    return out_df


In [ ]:
create_balanced_splits_cds_specific(
    mfa_root="/work/DISCOURSE/Data/Discourse/AUDIO_CHUNKED/TOPSY",
    metadata_path="/work/DISCOURSE/Data/Discourse/METADATA/TOPSY Metadata 2025.xlsx",
    output_csv="TOPSY_splits.csv",
    dataset_name="TOPSY",
    participant_col="participant_id",
    session_col="session_id")